# Developmental four-in-a-row: clean copy + stored-representation analysis (free, on Google Colab)

Data: Ma, Phaneuf, van Opheusden, Ma & Hartley (ages 8-25), public repo `github.com/ili-ma/fourinarow_analyses`.
This notebook downloads it into the temporary Colab machine, keeps **only** a clean copy (study ID, age to 0.1 year, gender, questionnaire totals, model parameters, moves), runs the analyses and saves to your Drive folder `compression-learner-data/fourinarow_dev/`. The raw download is deleted at the end; it contains identifying fields (IP addresses, coordinates, birth dates) that are never copied.

How to run: Runtime -> Run all (about 3 minutes). Code: `pipelines/fourinarow/` in the public repo.

In [ ]:
import os, subprocess, shutil
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/compression-learner-data/fourinarow_dev"
os.makedirs(OUT, exist_ok=True)
BASE = "https://raw.githubusercontent.com/saurabh9729/compression-learner-data/main/pipelines/fourinarow"
os.makedirs("/content/fiar_code", exist_ok=True); os.chdir("/content/fiar_code")
for f in ["clean_dev.py", "build_stores.py", "stats.py", "tests_dev.py"]:
    subprocess.run(["curl", "-sSfLO", f"{BASE}/{f}"], check=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/ili-ma/fourinarow_analyses", "/content/raw"], check=True)
print("downloaded")

In [ ]:
# Clean copy -> Drive, then delete the raw download
os.environ["FIAR_DIR"] = OUT
r = subprocess.run(["python", "clean_dev.py", "/content/raw/Analysis notebooks/data", OUT], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
shutil.rmtree("/content/raw", ignore_errors=True)
print("raw download deleted; Drive folder now has:", sorted(os.listdir(OUT)))

In [ ]:
# Stores and tests (results also saved as results_dev.txt)
for step in ["build_stores.py", "tests_dev.py"]:
    r = subprocess.run(["python", step], capture_output=True, text=True, env={**os.environ, "FIAR_DIR": OUT})
    print(r.stdout); print(r.stderr[-3000:])
    if step == "tests_dev.py":
        open(f"{OUT}/results_dev.txt", "w").write(r.stdout)
for f in ["clean_dev.py", "build_stores.py", "stats.py", "tests_dev.py"]:
    os.makedirs(f"{OUT}/code", exist_ok=True); shutil.copy(f, f"{OUT}/code/{f}")
print("saved to", OUT, sorted(os.listdir(OUT)))